# SymbolicTensors — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and their hierarchy 

---
## 1. Loading package

In [1]:
using SymbolicTensors

[ Info: Precompiling SymbolicTensors [55c8b40c-cbfa-4141-803e-4831c9841971](cache misses: include_dependency fsize change (1), mismatched flags (8))
[ Info: Precompiling SymbolicTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (2), mismatched flags (16))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension (concrete `Int` or symbolic `Symbol`)
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of all vector bundles over this manifold


In [3]:
?@def_manifold

```julia
@def_manifold <name> dim coord_indices frame_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frames.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

`dim` accepts a concrete positive integer or a bare symbol (e.g. `d`) for parametric / general-dimension calculations where the dimension is not fixed at definition time.

### Bindings in the caller's scope

  * `<name>`            → [`Manifold`](@ref) instance
  * `tangent<name>`     → [`VBundle`](@ref) (`isdual = false`)
  * `cotangent<name>`   → [`VBundle`](@ref) (`isdual = true`)
  * `frame<name>`       → [`FrameBundle`](@ref) (moving frame bundle)
  * `coframe<name>`     → [`FrameBundle`](@ref) (moving coframe bundle)
  * coordinate frame bindings `cf_<name>`, `ccf_<name>` (default; display as `∂`, `dx`)
  * moving frame bindings `mf_<name>`, `mcf_<name>` (default; display as `e`, `θ`)
  * each symbol in `coord_indices` → [`CoordinateIndex`](@ref) (contravariant)
  * each symbol in `frame_indices` → [`FrameIndex`](@ref) (contravariant)

### Index variance

Coordinate indices:

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Frame indices:

```julia
A1          # FrameIndex(:A1, :tangentM)   — contravariant
-A1         # FrameIndex(:A1, :cotangentM) — covariant
```

### Keyword arguments

| Keyword    | Default                                          | Description                                                                             |
|:---------- |:------------------------------------------------ |:--------------------------------------------------------------------------------------- |
| `frames`   | `[cf_<name>, ccf_<name>, mf_<name>, mcf_<name>]` | Caller-scope binding symbols (coord frame, coord coframe, moving frame, moving coframe) |
| `print_as` | `["∂", "dx", "e", "θ"]`                          | Display label strings for the four bases                                                |

All four entries in each vector must be distinct. Reusing a binding already registered for another manifold triggers a warning.

### Examples

```julia
# Minimal — default bindings cf_M, ccf_M, mf_M, mcf_M; display ∂, dx, e, θ
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
# Binds: M, tangentM, cotangentM, frameM, coframeM,
#        cf_M, ccf_M, mf_M, mcf_M,
#        a1..a4 (CoordinateIndex), A1..A4 (FrameIndex)
ccf_M[a1]   # displays as dx[a1]

# Parametric dimension
@def_manifold M d [a1, a2, a3, a4] [A1, A2, A3, A4]

# Custom bindings and display labels
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4] \
    frames=[:cfM, :ccfM, :mfM, :mcfM] \
    print_as=["∂", "dx", "e", "θ"]
```


In [4]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined coordinate frame ∂ (binding :cf_M) on tangentM and coordinate coframe dx (binding :ccf_M) on cotangentM
Defined frame bundle frameM (moving frame e, binding :mf_M) and coframe bundle coframeM (moving coframe θ, binding :mcf_M) over M


#### The different output for a struct like the manifold M 

In [5]:
# Jupyter output (MIME text/html)
M

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [6]:
# Build in repr of julia lang: representative string output ( string of MIME io default)
repr(M)

"Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])"

In [7]:
Meta.parse(Out[6])

:(Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM]))

In [8]:
eval(:(Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])))

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [9]:
# More efficient construction of Expr 
expr_form(M)

:(SymbolicTensors.Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM]))

In [10]:
@doc expr_form

```julia
expr_form(x)

Return an expression that reconstructs `x` via its constructor.
Useful for debugging and development: copy-paste the result to recreate objects.
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
expr_form(a1)
# :(CoordinateIndex(:a1, :tangentM))
expr_form(M)
# :(Manifold(...))  # all field values inlined
```


In [11]:
# dot access to a Manifold properties 
(M.name, M.dim, M.tangent_bundle, M.cotangent_bundle, M.vbundles)

(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])

In [12]:
# Manifold registry
(_MANIFOLDS,M==_MANIFOLDS[:M])

(Dict{Symbol, Manifold}(:M => Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])), true)

In [13]:
(expr_form(M),typeof(expr_form(M)))

(:(SymbolicTensors.Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])), Expr)

In [14]:
typeof(expr_form(M))

Expr

In [15]:
(expr_form(M).head,expr_form(M).args)

(:call, Any[:(SymbolicTensors.Manifold), :(:M), 4, :(:tangentM), :(:cotangentM), [:tangentM, :cotangentM]])

In [16]:
Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


### b. (co)Tangent bundle, (co)frames

In [17]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and by [`@def_vbundle`](@ref) for custom bundles. Bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name             # :tangentM
tangentM.manifold         # :M
tangentM.dim              # 4
tangentM.isdual           # false
tangentM.dual             # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.frame_indices       # [FrameIndex(:A1, :tangentM), ...]
tangentM.bases            # [Basis(:cf_M, :tangentM, :coordinate, "∂"),
                          #  Basis(:mf_M, :tangentM, :frame, "e")]
```

### Fields

  * `name`               : bundle name, e.g. `:tangentM`
  * `manifold`           : base manifold name, e.g. `:M`
  * `dim`                : fibre dimension
  * `isdual`             : `false` = primal (contravariant), `true` = dual                        (covariant). Authoritative for variance via                        [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`               : name of the paired dual bundle
  * `coordinate_indices` : [`CoordinateIndex`](@ref) objects for the coordinate                        chart; nonempty for tangent/cotangent bundles
  * `frame_indices`      : [`FrameIndex`](@ref) objects for the fibre frame;                        populated by [`@def_manifold`](@ref) and                        [`@def_vbundle`](@ref)
  * `bases`              : [`Basis`](@ref) objects for the coordinate and frame bases


#### (co)tangent space 

In [18]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(:cf_M, :tangentM, :coordinate, "∂"), Basis(:mf_M, :tangentM, :frame, "e")])

In [19]:
(tangentM.name,tangentM.manifold,tangentM.dim,tangentM.isdual
,tangentM.dual,tangentM.coordinate_indices,tangentM.frame_indices,tangentM.bases)

(:tangentM, :M, 4, false, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)], Basis[Basis(:cf_M, :tangentM, :coordinate, "∂"), Basis(:mf_M, :tangentM, :frame, "e")])

In [20]:
repr(tangentM)

"VBundle(:tangentM, :M, 4, false, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)])"

In [21]:
expr_form(tangentM)

:(SymbolicTensors.VBundle(:tangentM, :M, 4, false, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]))

In [22]:
repr(tangentM.frame_indices)

"FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]"

In [23]:
FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]==
[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]

true

#### Basis and BasisElement

In [24]:
@doc Basis

```julia
Basis
```

A named frame for a vector bundle, of a given type (`:coordinate` or `:frame`). Instances are created by [`@def_manifold`](@ref) (coordinate and frame) or standalone [`@def_frame_bundle`](@ref) (frame only for custom bundles).

### Fields

  * `name`     : binding symbol in the caller, e.g. `:cf_M`, `:ccf_M`
  * `vbundle`  : the bundle this frame is for, e.g. `:cotangentM`
  * `type`     : `:coordinate` (coordinate-induced) or `:frame` (arbitrary local frame)
  * `print_as` : display label string, e.g. `"dx"`, `"∂"`, `"e"`, `"θ"`

Index with the **binding** (`ccf_M[a1]`); REPL compact `show` uses `print_as`:

```julia
ccf_M[a1]    # a1 ∈ tangentM  → displays as dx[a1]
mcf_M[-A1]   # -A1 ∈ cotangentM → displays as θ[A1]
```


In [25]:
# Frames and coFrames 
(cf_M[-a1], ccf_M[a1], mf_M[-A1], mcf_M[A1])

(BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)))

In [26]:
cf_M

∂

In [27]:
expr_form(cf_M)

:(SymbolicTensors.Basis(:cf_M, :tangentM, :coordinate, "∂"))

In [28]:
SymbolicTensors.Basis(:cf_M, :tangentM, :coordinate, "∂")

∂

In [29]:
_BASES

Dict{Tuple{Symbol, Symbol}, Basis} with 4 entries:
  (:tangentM, :frame)        => Basis(:mf_M, :tangentM, :frame, "e")
  (:cotangentM, :frame)      => Basis(:mcf_M, :cotangentM, :frame, "θ")
  (:tangentM, :coordinate)   => Basis(:cf_M, :tangentM, :coordinate, "∂")
  (:cotangentM, :coordinate) => Basis(:ccf_M, :cotangentM, :coordinate, "dx")

In [64]:
tangentM.bases

2-element Vector{Basis}:
 ∂
 e

In [65]:
repr(tangentM.bases)

"Basis[Basis(:cf_M, :tangentM, :coordinate, \"∂\"), Basis(:mf_M, :tangentM, :frame, \"e\")]"

In [66]:
cotangentM.bases

2-element Vector{Basis}:
 dx
 θ

In [67]:
repr(cotangentM.bases)

"Basis[Basis(:ccf_M, :cotangentM, :coordinate, \"dx\"), Basis(:mcf_M, :cotangentM, :frame, \"θ\")]"

In [37]:
@doc BasisElement

```julia
BasisElement
```

A single element of a basis (coordinate or frame), constructed by `getindex` on a [`Basis`](@ref).

### Fields

  * `basis` : the [`Basis`](@ref) this element belongs to
  * `index` : the [`AbstractIndex`](@ref) labeling this element;           its vbundle is the **dual** of `basis.vbundle`

    ccf*M[a1] → BasisElement(Basis(:ccf*M, :cotangentM, :coordinate, "dx"), ...)   mcf*M[A1] → BasisElement(Basis(:mcf*M, :cotangentM, :frame, "θ"), ...)   cf*M[-a1] → BasisElement(Basis(:cf*M, :tangentM, :coordinate, "∂"), ...)   mf*M[-A1] → BasisElement(Basis(:mf*M, :tangentM, :frame, "e"), ...)


In [68]:
(ccf_M[a1],cf_M[-a1])

(BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)))

In [70]:
(mcf_M[A1],mf_M[-A1])

(BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)))

##### Error handling 

In [71]:
ccf_M[-a1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentM. Expected index from :tangentM. Index the basis binding :ccf_M (prints as dx).

In [72]:
ccf_M[A1]

LoadError: BasisElement: coordinate basis dx requires a CoordinateIndex (e.g. a1 or -a1); got FrameIndex.

#### Change of basis coordinate <-> moving

---
## 3. Tensors
---

### a. Type, definition and tensor components

In [44]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.


In [45]:
@doc @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as=:...] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All VBundle symbols must belong to the same manifold.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g M

@def_tensor T [cotangentM, cotangentM]                 # (0,2) tensor, metric auto-resolved
@def_tensor F [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T [tangentM, cotangentM]            # (1,1) mixed tensor
```


In [48]:
@def_tensor T [cotangentM, cotangentM]

┌ Warning: No metric is defined on manifold M. Tensor T has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:325


Defined tensor T on manifold M with 2 slot(s), metric=nothing


In [49]:
T

Manifold,M
Rank,2
Slots,↓cotangentM ↓cotangentM
Symmetries,"SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])"
Traceless,false
Metric,none


In [50]:
(T.is_traceless,T.known_traces,T.manifold,T.metric,T.print_as,T.rank,T.slots,T.symmetries)

(false, Any[], :M, nothing, :T, 2, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])])

In [51]:
T.symmetries

1-element Vector{SlotSymmetry}:
 NoSymmetry(n=2)

In [52]:
T.slots

2-element Vector{Symbol}:
 :cotangentM
 :cotangentM

In [53]:
T[-a1,-a2]

T[-a1, -a2]

In [54]:
(T[-a1,-a2],T[a1,-a2],T[-a1,a2],T[a1,a2],T[-A1,-A2],T[A1,-a1],T[A1,a1])

(TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :cotangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :tangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[FrameIndex(:A1, :tangentM), CoordinateIndex(:a1, :cotangentM)]), TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[FrameIndex(:A1, :tangentM), CoordinateIndex(:a1, :tangentM)]))

##### Error handling 

In [55]:
T[-a1,-a1]

T[-a1, -a1]

In [56]:
T[a1,a1]

T[a1, a1]

### b. Expansion of a tensor within a basis 

In [57]:
@doc basis_expansion

```julia
basis_expansion(T::Tensor, style::ExpansionStyle=Coordinate) -> BasisExpansion
basis_expansion(T::Tensor) -> BasisExpansion
```

Expand tensor `T` slot-by-slot using canonical indices and the given [`ExpansionStyle`](@ref).

  * **`Coordinate`** (default): per slot use `:coordinate` basis if registered, else `:frame`.
  * **`Frame`**: per slot use `:frame` basis only.

# Examples

```julia
basis_expansion(T)             # Coordinate style
basis_expansion(T, Coordinate)
basis_expansion(T, Frame)
```


In [58]:
basis_expansion(T)

TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)]) BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)) ⊗ BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))

In [59]:
(basis_expansion(T, Coordinate),basis_expansion(T, Frame))

(BasisExpansion(TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)]), BasisElement[BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorExpression(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :T, nothing), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)]), BasisElement[BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))

In [60]:
@def_tensor B [tangentM, cotangentM]

Defined tensor B on manifold M with 2 slot(s), metric=nothing


┌ Warning: No metric is defined on manifold M. Tensor B has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:325


In [61]:
(basis_expansion(B,Coordinate),basis_expansion(B, Frame))

(BasisExpansion(TensorExpression(Tensor(:M, [:tangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :B, nothing), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :cotangentM)]), BasisElement[BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorExpression(Tensor(:M, [:tangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], :B, nothing), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :cotangentM)]), BasisElement[BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))